# PhysVLM Colab Pro Reproduction

这个 notebook 面向 Colab Pro：优先使用 L4 / A100，拿不到再用 T4。目标是跑通 PhysVLM 的独立推理与离线 benchmark 评测，不依赖官方缺失的 `start_physvlm_server.py`。

建议流程：先运行到模型加载 smoke test，再跑 1 条样例推理，最后再跑完整 EQA-phys benchmark。

## 0. Runtime 选择

在 Colab 顶部菜单选择：`Runtime -> Change runtime type -> GPU`。

推荐顺序：

1. `A100`：可尝试 fp16，不需要 4bit。
2. `L4`：优先推荐，通常 4bit 更稳。
3. `T4`：可用，但建议必须 4bit。

下面的单元会打印 GPU 和显存，并自动给出 `LOAD_4BIT` 建议。

In [ ]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path('/content')
PHYSVLM_REPO = PROJECT_ROOT / 'PhysVLM'
PHYSVLM_ROOT = PHYSVLM_REPO / 'physvlm-main'
REPRO_ROOT = PROJECT_ROOT / 'PhysVLM-reproduce'

print(subprocess.check_output(['nvidia-smi'], text=True))


In [ ]:
import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
LOAD_4BIT = gpu_mem_gb < 24
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print({'gpu': gpu_name, 'vram_gb': round(gpu_mem_gb, 2), 'device': DEVICE, 'recommended_load_4bit': LOAD_4BIT})


## 1. 安装依赖

这里不使用 `pip install -e .` 安装官方包，避免它把 Colab 默认环境改乱。我们直接把官方代码目录加入 `PYTHONPATH`，并手动安装 PhysVLM 需要的核心依赖。

如果切换 runtime，需要重新运行本节。

In [ ]:
!pip -q install \
  transformers==4.37.2 \
  tokenizers==0.15.1 \
  accelerate==0.21.0 \
  peft==0.10.0 \
  sentencepiece==0.1.99 \
  einops==0.6.1 \
  einops-exts==0.0.4 \
  timm==0.6.13 \
  shortuuid \
  huggingface-hub \
  opencv-python-headless \
  pybullet \
  tqdm

# 仅 CUDA 4bit 量化需要。Colab Pro 上不固定版本更容易适配当前 CUDA 镜像。
!pip -q install bitsandbytes


## 2. 获取官方代码并应用补丁

官方仓库当前有两个复现阻塞点：

- full Hugging Face checkpoint 加载分支提前 `return None`。
- released config 里写的是 `Siglip/siglip-so400m-patch14-384`，实际公开模型是 `google/siglip-so400m-patch14-384`。

下面单元会 clone 官方仓库并做最小补丁。

In [ ]:
import subprocess

if not PHYSVLM_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/unira-zwj/PhysVLM.git', str(PHYSVLM_REPO)], check=True)
else:
    print(f'Using existing repo: {PHYSVLM_REPO}')

os.environ['PYTHONPATH'] = f"{PHYSVLM_ROOT}:{os.environ.get('PYTHONPATH', '')}"
print('PHYSVLM_ROOT =', PHYSVLM_ROOT)


In [ ]:
from pathlib import Path

builder = PHYSVLM_ROOT / 'physvlm/model/builder.py'
text = builder.read_text()

# Patch 1: Remove "return None" after full-checkpoint loading if present.
original_text = text
text = text.replace(
    '        print("---model---\n", tokenizer)\n        print("---model---\n", model)\n        return None\n',
    '        print("---model---\n", tokenizer)\n        print("---model---\n", model)\n',
)
text = text.replace(
    '        print("---model---\n", model)\n        return None\n',
    '        print("---model---\n", model)\n',
)
# Verify patch result
if 'return None' in text:
    print('WARNING: "return None" still present in builder.py after patch — check upstream changes.')
if original_text == text:
    print('INFO: builder.py patch was a no-op (upstream already fixed). Continuing.')

builder.write_text(text)

encoder_builder = PHYSVLM_ROOT / 'physvlm/model/multimodal_encoder/builder.py'
text = encoder_builder.read_text()
if '_resolve_siglip_alias' not in text:
    text = text.replace(
        'from .siglip_encoder import SigLipVisionTower\n\n\n',
        'from .siglip_encoder import SigLipVisionTower\n\n\ndef _resolve_siglip_alias(tower_name):\n    if tower_name == "Siglip/siglip-so400m-patch14-384":\n        return "google/siglip-so400m-patch14-384"\n    return tower_name\n\n\n',
    )
    text = text.replace(
        "vision_tower = getattr(vision_tower_cfg, 'mm_vision_tower', getattr(vision_tower_cfg, 'vision_tower', None))",
        "vision_tower = _resolve_siglip_alias(getattr(vision_tower_cfg, 'mm_vision_tower', getattr(vision_tower_cfg, 'vision_tower', None)))",
    )
    text = text.replace(
        "depth_tower = getattr(depth_tower_cfg, 'mm_depth_tower', getattr(depth_tower_cfg, 'depth_tower', None))",
        "depth_tower = _resolve_siglip_alias(getattr(depth_tower_cfg, 'mm_depth_tower', getattr(depth_tower_cfg, 'depth_tower', None)))",
    )
    text = text.replace(
        "vision_tower.startswith(\"google\") or vision_tower.startswith('bczhou')",
        "\"/\" in vision_tower",
    )
    text = text.replace(
        "depth_tower.startswith(\"google\") or depth_tower.startswith('bczhou')",
        "\"/\" in depth_tower",
    )
    encoder_builder.write_text(text)
    print('Patched encoder builder.py (siglip alias + tower detection).')
else:
    print('Encoder builder.py already patched. Skipping.')

print('Patch step complete.')

## 3. 写入 standalone 脚本

这两个脚本就是本地仓库里的核心交付物：

- `standalone_inference.py`：直接调用模型推理。
- `eval_standalone.py`：离线跑 EQA-phys JSON 评测。

In [ ]:
REPRO_ROOT.mkdir(parents=True, exist_ok=True)
(REPRO_ROOT / 'scripts').mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile /content/PhysVLM-reproduce/scripts/standalone_inference.py
from __future__ import annotations

import argparse
import json
import re
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

from PIL import Image, ImageDraw


DEFAULT_MODEL_NAME = "physvlm-qwen2"
DEFAULT_CONV_MODE = "qwen2"
DEFAULT_SYSTEM_PROMPT_PREFIX = "<image>\n<depth>\n"


@dataclass
class Prediction:
    image_path: str
    depth_path: str | None
    question: str
    answer: str
    prompt: str
    model_path: str


def default_physvlm_root() -> Path:
    return Path(__file__).resolve().parents[2] / "github_repos" / "PhysVLM" / "physvlm-main"


def add_physvlm_to_path(physvlm_root: str | Path) -> Path:
    root = Path(physvlm_root).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"PhysVLM root does not exist: {root}")
    sys.path.insert(0, str(root))
    return root


def choose_device(device: str) -> str:
    if device != "auto":
        return device

    import torch

    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def load_rgb(path: str | Path) -> Image.Image:
    return Image.open(path).convert("RGB")


def black_depth_like(image: Image.Image) -> Image.Image:
    return Image.new("RGB", image.size, (0, 0, 0))


def normalize_stop_text(text: str, stop_str: str | None) -> str:
    output = text.strip()
    if stop_str and output.endswith(stop_str):
        output = output[: -len(stop_str)].strip()
    if "ASSISTANT:" in output:
        output = output.split("ASSISTANT:", 1)[-1].strip()
    return output


def parse_bbox(answer: str, image_size: tuple[int, int]) -> tuple[int, int, int, int] | None:
    match = re.fullmatch(
        r"\[\s*([-+]?\d*\.?\d+)\s*,\s*([-+]?\d*\.?\d+)\s*,\s*([-+]?\d*\.?\d+)\s*,\s*([-+]?\d*\.?\d+)\s*\]",
        answer.strip(),
    )
    if not match:
        return None

    values = [float(v) for v in match.groups()]
    width, height = image_size
    if all(0.0 <= v <= 1.0 for v in values):
        values = [values[0] * width, values[1] * height, values[2] * width, values[3] * height]

    x1, y1, x2, y2 = [round(v) for v in values]
    x1, x2 = sorted((max(0, min(width - 1, x1)), max(0, min(width - 1, x2))))
    y1, y2 = sorted((max(0, min(height - 1, y1)), max(0, min(height - 1, y2))))
    if x2 <= x1 or y2 <= y1:
        return None
    return x1, y1, x2, y2


def save_visualization(image_path: str | Path, answer: str, output_path: str | Path) -> None:
    image = load_rgb(image_path)
    bbox = parse_bbox(answer, image.size)
    if bbox is not None:
        draw = ImageDraw.Draw(image)
        draw.rectangle(bbox, outline=(0, 255, 0), width=4)
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    image.save(output)


class PhysVLMPredictor:
    def __init__(
        self,
        model_path: str | Path,
        physvlm_root: str | Path | None = None,
        model_base: str | None = None,
        model_name: str = DEFAULT_MODEL_NAME,
        conv_mode: str = DEFAULT_CONV_MODE,
        device: str = "auto",
        load_8bit: bool = False,
        load_4bit: bool = False,
        use_flash_attn: bool = False,
    ) -> None:
        root = add_physvlm_to_path(physvlm_root or default_physvlm_root())
        self.physvlm_root = root
        self.model_path = str(Path(model_path).expanduser())
        self.model_base = model_base
        self.model_name = model_name
        self.conv_mode = conv_mode
        self.device = choose_device(device)

        if self.device != "cuda" and (load_8bit or load_4bit):
            raise ValueError("8-bit/4-bit loading is only supported on CUDA in this reproduction script.")

        import torch
        from physvlm.conversation import SeparatorStyle, conv_templates
        from physvlm.model.builder import load_pretrained_model

        self.torch = torch
        self.SeparatorStyle = SeparatorStyle
        self.conv_templates = conv_templates

        loaded = load_pretrained_model(
            model_path=self.model_path,
            model_base=self.model_base,
            model_name=self.model_name,
            load_8bit=load_8bit,
            load_4bit=load_4bit,
            device=self.device,
            use_flash_attn=use_flash_attn,
        )
        if loaded is None:
            raise RuntimeError(
                "load_pretrained_model returned None. Apply the local builder.py compatibility patch first."
            )

        self.tokenizer, self.model, self.image_processor, self.context_len = loaded
        self.model.eval()

    def build_prompt(self, question: str) -> tuple[str, str | None]:
        conv = self.conv_templates[self.conv_mode].copy()
        user_message = DEFAULT_SYSTEM_PROMPT_PREFIX + question.strip()
        conv.append_message(conv.roles[0], user_message)
        conv.append_message(conv.roles[1], None)
        stop_str = conv.sep if conv.sep_style != self.SeparatorStyle.TWO else conv.sep2
        return conv.get_prompt(), stop_str

    def preprocess_pair(
        self,
        image_path: str | Path,
        depth_path: str | Path | None,
    ) -> tuple[Any, Any, list[tuple[int, int]], Image.Image]:
        from physvlm.mm_utils import process_images

        image = load_rgb(image_path)
        depth = load_rgb(depth_path) if depth_path else black_depth_like(image)
        image_sizes = [image.size]

        image_tensor = process_images([image], self.image_processor, self.model.config)
        depth_tensor = process_images([depth], self.image_processor, self.model.config)

        dtype = next(self.model.parameters()).dtype
        device = self.model.device

        if isinstance(image_tensor, list):
            image_tensor = [item.to(device=device, dtype=dtype) for item in image_tensor]
        else:
            image_tensor = image_tensor.to(device=device, dtype=dtype)

        if isinstance(depth_tensor, list):
            depth_tensor = [item.to(device=device, dtype=dtype) for item in depth_tensor]
        else:
            depth_tensor = depth_tensor.to(device=device, dtype=dtype)

        return image_tensor, depth_tensor, image_sizes, image

    def predict(
        self,
        image_path: str | Path,
        question: str,
        depth_path: str | Path | None = None,
        temperature: float = 0.0,
        top_p: float = 1.0,
        max_new_tokens: int = 128,
    ) -> Prediction:
        from physvlm.constants import IMAGE_TOKEN_INDEX
        from physvlm.mm_utils import tokenizer_image_token

        prompt, stop_str = self.build_prompt(question)
        image_tensor, depth_tensor, image_sizes, _ = self.preprocess_pair(image_path, depth_path)

        input_ids = tokenizer_image_token(
            prompt,
            self.tokenizer,
            IMAGE_TOKEN_INDEX,
            return_tensors="pt",
        ).unsqueeze(0).to(self.model.device)

        do_sample = temperature > 0.001
        generation_kwargs: dict[str, Any] = {
            "inputs": input_ids,
            "images": image_tensor,
            "depth_images": depth_tensor,
            "image_sizes": image_sizes,
            "do_sample": do_sample,
            "top_p": top_p,
            "max_new_tokens": max_new_tokens,
            "use_cache": True,
        }
        if do_sample:
            generation_kwargs["temperature"] = temperature

        with self.torch.inference_mode():
            output_ids = self.model.generate(**generation_kwargs)

        answer = self.tokenizer.decode(output_ids[0], skip_special_tokens=True)
        answer = normalize_stop_text(answer, stop_str)
        return Prediction(
            image_path=str(image_path),
            depth_path=str(depth_path) if depth_path else None,
            question=question,
            answer=answer,
            prompt=prompt,
            model_path=self.model_path,
        )


def write_json(path: str | Path, payload: dict[str, Any]) -> None:
    output = Path(path)
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Standalone PhysVLM inference without FastAPI server.")
    parser.add_argument("--model-path", required=True)
    parser.add_argument("--model-base", default=None)
    parser.add_argument("--model-name", default=DEFAULT_MODEL_NAME)
    parser.add_argument("--physvlm-root", default=str(default_physvlm_root()))
    parser.add_argument("--conv-mode", default=DEFAULT_CONV_MODE)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--load-8bit", action="store_true")
    parser.add_argument("--load-4bit", action="store_true")
    parser.add_argument("--use-flash-attn", action="store_true")
    parser.add_argument("--image-path", required=True)
    parser.add_argument("--depth-path", default=None)
    parser.add_argument("--question", required=True)
    parser.add_argument("--temperature", type=float, default=0.0)
    parser.add_argument("--top-p", type=float, default=1.0)
    parser.add_argument("--max-new-tokens", type=int, default=128)
    parser.add_argument("--output-json", default=None)
    parser.add_argument("--visualization-path", default=None)
    return parser


def main() -> None:
    args = build_arg_parser().parse_args()
    predictor = PhysVLMPredictor(
        model_path=args.model_path,
        physvlm_root=args.physvlm_root,
        model_base=args.model_base,
        model_name=args.model_name,
        conv_mode=args.conv_mode,
        device=args.device,
        load_8bit=args.load_8bit,
        load_4bit=args.load_4bit,
        use_flash_attn=args.use_flash_attn,
    )
    prediction = predictor.predict(
        image_path=args.image_path,
        depth_path=args.depth_path,
        question=args.question,
        temperature=args.temperature,
        top_p=args.top_p,
        max_new_tokens=args.max_new_tokens,
    )

    print(prediction.answer)
    if args.output_json:
        write_json(args.output_json, asdict(prediction))
    if args.visualization_path:
        save_visualization(args.image_path, prediction.answer, args.visualization_path)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/PhysVLM-reproduce/scripts/eval_standalone.py
from __future__ import annotations

import argparse
import importlib.util
import json
import sys
from collections import defaultdict
from pathlib import Path
from typing import Any

# Dynamic import so the script works regardless of cwd or sys.path.
_SCRIPT_DIR = Path(__file__).resolve().parent
_spec = importlib.util.spec_from_file_location(
    "standalone_inference", _SCRIPT_DIR / "standalone_inference.py"
)
_mod = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = _mod
_spec.loader.exec_module(_mod)

DEFAULT_CONV_MODE = _mod.DEFAULT_CONV_MODE
DEFAULT_MODEL_NAME = _mod.DEFAULT_MODEL_NAME
PhysVLMPredictor = _mod.PhysVLMPredictor
default_physvlm_root = _mod.default_physvlm_root
write_json = _mod.write_json


def read_json(path: str | Path) -> Any:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def resolve_data_path(data_root: str | Path | None, maybe_relative: str | None) -> str | None:
    if not maybe_relative:
        return None
    candidate = Path(maybe_relative).expanduser()
    if candidate.is_absolute():
        return str(candidate)
    if data_root is None:
        return str(candidate)
    return str(Path(data_root).expanduser() / candidate)


def robot_name(row: dict[str, Any]) -> str:
    explicit = row.get("robot") or row.get("robot_name")
    if explicit:
        return str(explicit).upper()

    haystack = " ".join(str(row.get(key, "")) for key in ("image", "depth", "scene", "id")).upper()
    for name in ("UR5", "CR5", "FR5", "PANDA", "UR3", "XARM6"):
        if name in haystack:
            return name
    return "UNKNOWN"


def normalize_answer(answer: Any) -> str:
    return str(answer).strip().lower()


def _first_word(text: str) -> str:
    """Extract the first word, stripping punctuation."""
    word = normalize_answer(text).split()[0] if normalize_answer(text).split() else ""
    return word.rstrip(".,;:!?")


def is_correct(prediction: str, label: str) -> bool:
    return normalize_answer(prediction)[:3] == normalize_answer(label)[:3]


def is_strict_correct(prediction: str, label: str) -> bool:
    return _first_word(prediction) == _first_word(label)


def summarize(predictions: list[dict[str, Any]]) -> dict[str, Any]:
    totals: dict[str, int] = defaultdict(int)
    corrects: dict[str, int] = defaultdict(int)

    for item in predictions:
        robot = item["robot"]
        totals[robot] += 1
        totals["ALL"] += 1
        if item["correct"]:
            corrects[robot] += 1
            corrects["ALL"] += 1
        if item.get("strict_correct", False):
            corrects[f"{robot}_STRICT"] += 1
            corrects["ALL_STRICT"] += 1

    metrics = {}
    for robot in sorted(totals):
        total = totals[robot]
        correct = corrects[robot]
        metrics[robot] = {
            "correct": correct,
            "total": total,
            "accuracy": round(correct / total, 4) if total else 0.0,
            "strict_correct": corrects[f"{robot}_STRICT"],
            "strict_accuracy": round(corrects[f"{robot}_STRICT"] / total, 4) if total else 0.0,
        }
    return metrics


def evaluate(args: argparse.Namespace) -> dict[str, Any]:
    rows = read_json(args.qa_json)
    if not isinstance(rows, list):
        raise ValueError("QA JSON must contain a list of examples.")

    predictor = PhysVLMPredictor(
        model_path=args.model_path,
        physvlm_root=args.physvlm_root,
        model_base=args.model_base,
        model_name=args.model_name,
        conv_mode=args.conv_mode,
        device=args.device,
        load_8bit=args.load_8bit,
        load_4bit=args.load_4bit,
        use_flash_attn=args.use_flash_attn,
    )

    selected = rows[: args.limit] if args.limit else rows
    predictions: list[dict[str, Any]] = []

    for index, row in enumerate(selected):
        image_path = resolve_data_path(args.data_root, row.get("image") or row.get("image_path"))
        depth_path = resolve_data_path(args.data_root, row.get("depth") or row.get("depth_path"))
        question = row.get("question") or row.get("query")
        label = row.get("answer") or row.get("label")

        if not image_path or not question or label is None:
            raise ValueError(f"Example {index} is missing image/question/answer fields: {row}")

        prediction = predictor.predict(
            image_path=image_path,
            depth_path=depth_path,
            question=str(question),
            temperature=args.temperature,
            top_p=args.top_p,
            max_new_tokens=args.max_new_tokens,
        )

        pred_row = {
            "index": index,
            "robot": robot_name(row),
            "image": image_path,
            "depth": depth_path,
            "question": question,
            "label": label,
            "prediction": prediction.answer,
            "correct": is_correct(prediction.answer, str(label)),
            "strict_correct": is_strict_correct(prediction.answer, str(label)),
            "raw": row,
        }
        predictions.append(pred_row)

    return {
        "qa_json": str(args.qa_json),
        "data_root": str(args.data_root) if args.data_root else None,
        "model_path": str(args.model_path),
        "num_examples": len(predictions),
        "metrics": summarize(predictions),
        "predictions": predictions,
    }


def build_arg_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Offline PhysVLM EQA-phys evaluation.")
    parser.add_argument("--model-path", required=True)
    parser.add_argument("--model-base", default=None)
    parser.add_argument("--model-name", default=DEFAULT_MODEL_NAME)
    parser.add_argument("--physvlm-root", default=str(default_physvlm_root()))
    parser.add_argument("--conv-mode", default=DEFAULT_CONV_MODE)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--load-8bit", action="store_true")
    parser.add_argument("--load-4bit", action="store_true")
    parser.add_argument("--use-flash-attn", action="store_true")
    parser.add_argument("--qa-json", required=True)
    parser.add_argument("--data-root", default=None)
    parser.add_argument("--limit", type=int, default=0)
    parser.add_argument("--temperature", type=float, default=0.0)
    parser.add_argument("--top-p", type=float, default=1.0)
    parser.add_argument("--max-new-tokens", type=int, default=64)
    parser.add_argument("--output-json", required=True)
    return parser


def main() -> None:
    args = build_arg_parser().parse_args()
    result = evaluate(args)
    write_json(args.output_json, result)
    print(json.dumps(result["metrics"], ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


## 4. 下载模型

如果 Google Drive 空间够，建议把 checkpoint 放到 Drive，避免 Colab runtime 被回收后重复下载 8GB+ 权重。

如果 Drive I/O 太慢，把 `USE_DRIVE = False`，先下载到 `/content/checkpoints`。

In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_ROOT = Path('/content/drive/MyDrive/physvlm/checkpoints')
else:
    CHECKPOINT_ROOT = Path('/content/checkpoints')

MODEL_DIR = CHECKPOINT_ROOT / 'PhysVLM-Qwen2.5-3B'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('MODEL_DIR =', MODEL_DIR)


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id='JettZhou/PhysVLM-Qwen2.5-3B',
    local_dir=str(MODEL_DIR),
)
print('Model snapshot ready:', MODEL_DIR)


## 5. 模型加载 smoke test

先只验证模型能加载，不跑完整 benchmark。

- L4/T4：保留 `LOAD_4BIT=True`。
- A100：可以手动把 `LOAD_4BIT=False` 改掉测试 fp16。

In [ ]:
import sys
sys.path.insert(0, str(REPRO_ROOT / 'scripts'))

from standalone_inference import PhysVLMPredictor

predictor = PhysVLMPredictor(
    model_path=MODEL_DIR,
    physvlm_root=PHYSVLM_ROOT,
    device=DEVICE,
    load_4bit=LOAD_4BIT,
    use_flash_attn=False,
)
print('Loaded context_len:', predictor.context_len)
print('Vision tower:', type(predictor.model.get_vision_tower()).__name__)
print('Depth tower:', type(predictor.model.get_depth_tower()).__name__)


## 6. 单条推理

把 `IMAGE_PATH` 和 `DEPTH_PATH` 改成你生成的 RGB 图和 S-P Map。

如果只是验证推理链路，可以先令 `DEPTH_PATH = None`，脚本会用黑图占位；这只适合 smoke test，不适合正式结果。

In [ ]:
IMAGE_PATH = '/content/example_rgb.png'
DEPTH_PATH = None  # '/content/example_sp_map.png'
QUESTION = 'Can the robot reach the object on the table?'

prediction = predictor.predict(
    image_path=IMAGE_PATH,
    depth_path=DEPTH_PATH,
    question=QUESTION,
    temperature=0.0,
    max_new_tokens=64,
)
print(prediction.answer)


## 7. 生成 EQA-phys 数据（可选）

官方 simulator 的流程是：

```bash
python main.py --robot UR5 --dataset val
python generate_sp_map.py --robot UR5
python generate_qas.py
```

建议先只生成 `UR5` 小规模数据，确认路径和格式，再扩展到 `CR5 / FR5 / PANDA`。

In [ ]:
SIM_ROOT = PHYSVLM_REPO / 'EQA-phys-simulator'
print('SIM_ROOT =', SIM_ROOT)

# 示例：需要时取消注释运行。
# %cd /content/PhysVLM/EQA-phys-simulator
# !python main.py --robot UR5 --dataset val
# !python generate_sp_map.py --robot UR5
# !python generate_qas.py


## 8. 离线 benchmark 评测

`QA_JSON` 指向 `phys_bench_sim_qas.json`，`DATA_ROOT` 指向 JSON 中相对路径的根目录。先用 `--limit 20` 做小样本检查，再去掉 limit 跑完整评测。

In [ ]:
QA_JSON = '/content/PhysVLM/EQA-phys-simulator/phys_bench_sim_qas.json'
DATA_ROOT = '/content/PhysVLM/EQA-phys-simulator'
OUTPUT_JSON = '/content/physvlm_eval_results_limit20.json'

cmd = [
    'python', str(REPRO_ROOT / 'scripts/eval_standalone.py'),
    '--physvlm-root', str(PHYSVLM_ROOT),
    '--model-path', str(MODEL_DIR),
    '--qa-json', QA_JSON,
    '--data-root', DATA_ROOT,
    '--output-json', OUTPUT_JSON,
    '--device', DEVICE,
    '--limit', '20',
]
if LOAD_4BIT:
    cmd.append('--load-4bit')

print(' '.join(cmd))
# subprocess.run(cmd, check=True)


## 9. 结果保存

完整评测后，把以下文件下载或复制到 Drive：

- `physvlm_eval_results_limit20.json` 或完整评测 JSON。
- 可视化图片目录。
- 运行日志，包括 GPU 类型、是否 4bit、依赖版本。

这些会用于最终 README、简历项目描述和论文结果对比表。